# Download and enrich ChoiceAssay feeding detection data

This notebook: 
- downloads journal CSVs from blob storage (or uses local files*)
- aggregates them into one DataFrame
- interprets locations in L/R tube data
- enriches the data with subject/treatment info
- saves as a new aggregated CSV for future use (./downloads/aggregated_CAPOSE.csv)

*This processing assumes the journals are in the local ./downloads/src_journals directory.  You can either copy them their manually, or have this tool download them from the blobstore.  To automatically download, you will need the keys_choiceassay.env file in your user home /.expidite/ directory.

In [1]:
# Imports
from pathlib import Path
import pandas as pd
from expidite_rpi.core import configuration as root_cfg
from expidite_rpi.core.cloud_connector import CloudConnector

Logging expidite to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260731T125157850\logs\default_20260731T125157853.log at level 20
2026-07-31 13:51:57,855 expidite INFO   [4076] Loading C:\Users\bee-ops\.expidite\system.cfg...


In [ ]:
# Configuration
DOWNLOAD_FROM_AZURE = True
AZURE_KEYS_FILE = Path.home() / ".expidite" / "keys_choiceassay.env"
if DOWNLOAD_FROM_AZURE:
    assert AZURE_KEYS_FILE.exists(), f"Missing required config file: {AZURE_KEYS_FILE}"

MAX_X = 800
MAX_Y = 608

CONTAINER_NAME = "expidite-journals-reprocessed" # Use the reprocessed journals
TYPE_ID = "CAPOSE"
PREFIX = f"V3_{TYPE_ID}"
SUFFIX = ".csv"

# Local download directory (relative to notebook working directory)
DOWNLOAD_DIR = Path("./downloads")
SRC_JOURNAL_DIR = DOWNLOAD_DIR / "src_journals"
SRC_JOURNAL_DIR.mkdir(parents=True, exist_ok=True)
aggregated_csv_file = DOWNLOAD_DIR / f"aggregated_{TYPE_ID}.csv"

SUBJECTS_FILE = Path("./subjects.csv")
assert SUBJECTS_FILE.exists(), f"Missing required subjects file: {SUBJECTS_FILE}"

print(f"Downloading {PREFIX} files from '{CONTAINER_NAME}' to {DOWNLOAD_DIR.resolve()}")

## Download, combine, enrich and save journals (CSVs)

Either pull from Azure or use files in the local ./download directory

In [3]:
# Load the subjects.csv
def load_subjects(file_path=SUBJECTS_FILE):
    if not file_path.exists():
        msg = f"Subjects file {file_path} does not exist."
        raise FileNotFoundError(msg)
    subjects_df = pd.read_csv(file_path)

    # Create a datetime column for the exp_start_time and exp_end_time columns by combining the Date (dd/mm/yyyy) and Start_t and Stop_t columns (hh:mm)
    # The Stop_t is 1 day after the Start_t, so we need to add 1 day to the Date column when calculating the exp_end_time
    subjects_df["exp_start_time"] = pd.to_datetime(
        subjects_df["Date"] + " " + subjects_df["Start_t"], format="%d/%m/%Y %H:%M", utc=True
    )
    subjects_df["exp_end_time"] = pd.to_datetime(
        subjects_df["Date"] + " " + subjects_df["Stop_t"], format="%d/%m/%Y %H:%M", utc=True
    ) + pd.Timedelta(days=1)

    # We use the RPi column as the index to match into the journal data; check they're all valid integer values
    if not subjects_df["RPi"].apply(lambda x: str(x).isdigit()).all():
        msg = "Invalid RPi values in subjects file. All RPi values must be integers."
        raise ValueError(msg)
    subjects_df["RPi"] = subjects_df["RPi"].astype(int)

    # Check all values of feeding_from_tube are either "L" or "R"
    if not subjects_df["Tube"].isin(["L", "R"]).all():
        msg = "Invalid feeding_from_tube values in subjects file. All feeding_from_tube values must be 'L' or 'R'."
        raise ValueError(msg)

    # Print the first and last exp_start_time and exp_end_time for each RPi to check the ranges
    print("Subjects file loaded successfully. RPi experiment time ranges:")
    for rpi, group in subjects_df.groupby("RPi"):
        start_time = group["exp_start_time"].min()
        end_time = group["exp_end_time"].max()
        print(f"RPi {rpi}: exp_start_time: {start_time}, exp_end_time: {end_time}")

    return subjects_df

subjects_df = load_subjects()

def create_subjects_lookup_structure(subjects_df: pd.DataFrame):
    # Create a lookup dictionary for the subjects info by RPi and timestamp
    # There are multiple rows for each RPi, so we need to check the timestamp against the
    # exp_start_time and exp_end_time for each row
    subjects_lookup = {}
    for _, row in subjects_df.iterrows():
        rpi = row["RPi"]
        start_time = row["exp_start_time"]
        end_time = row["exp_end_time"]
        if rpi not in subjects_lookup:
            subjects_lookup[rpi] = {"L": [], "R": []}
        subjects_lookup[rpi][row["Tube"]].append({
            "start_time": start_time,
            "end_time": end_time,
            "subject_info": row.to_dict(),
        })
    return subjects_lookup

SUBJECTS_LOOKUP = create_subjects_lookup_structure(subjects_df)

def get_subject_info(rpi: int, tube: str, timestamp: pd.Timestamp):
    # Get the subject info for a given RPi and timestamp
    # We check that the timestamp is within the exp_start_time and exp_end_time for the RPi
    # If it is, we return the subject info as a dictionary; if not, we return None
    if rpi not in SUBJECTS_LOOKUP:
        msg = f"Invalid RPi value: {rpi}"
        raise ValueError(msg)

    if tube not in ["L", "R"]:
        msg = f"Invalid tube value: {tube}. Must be 'L' or 'R'."
        raise ValueError(msg)

    rpi_infos = SUBJECTS_LOOKUP[rpi][tube]
    for rpi_info in rpi_infos:
        if rpi_info["start_time"] <= timestamp <= rpi_info["end_time"]:
            return rpi_info["subject_info"]
    return None


Subjects file loaded successfully. RPi experiment time ranges:
RPi 1: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 2: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 3: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 4: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 5: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 6: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 7: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 8: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 9: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 10: exp_start_time: 2026-05-06 14:34:00+00:00, exp_end_time: 2026-06-18 14:45:00+00:00
RPi 11: exp_start_time: 2026-05-07 1

In [4]:
def download_files_from_blobstore():
    cc = CloudConnector.get_instance(root_cfg.CloudType.AZURE)
    cc.set_keys(AZURE_KEYS_FILE)
    files = cc.list_cloud_files(CONTAINER_NAME, prefix=PREFIX, suffix=SUFFIX)

    print(f"Found {len(files)} matching CSV files in blobstore")
    if files:
        cc.download_container(src_container=CONTAINER_NAME, dst_dir=SRC_JOURNAL_DIR, files=files, overwrite=False)
        print(f"Downloaded {len(files)} files to {SRC_JOURNAL_DIR.resolve()}")
    else:
        print("No matching files to download.")

if DOWNLOAD_FROM_AZURE:
    download_files_from_blobstore()

In [5]:
# Aggregate CSV journals
def aggregate_journals():
    csv_paths = sorted(SRC_JOURNAL_DIR.rglob("V3_CAPOSE_*.csv"))
    print(f"CSV files available locally: {len(csv_paths)}")

    required_fields = [
        "device_id",
        "timestamp",
        #"device_name",
        "Tube_prob_x",
        "Tube_prob_y",
        "Tube_prob_conf",
        "L_tube_x",
        "L_tube_y",
        "L_tube_conf",
        "R_tube_x",
        "R_tube_y",
        "R_tube_conf",
        "End_prob_x",
        "End_prob_y",
        "End_prob_conf",
        "source_filename",
        "frame_index",
        "frame_start_time",
    ]

    total_video_count = 0
    total_videos_with_tube_conf = 0
    df_list = []
    for i, csv_path in enumerate(csv_paths):
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"{i + 1}/{len(csv_paths)} Error reading {csv_path}: {e}")
            continue
        if not df.empty:
            if not all(field in df.columns for field in required_fields):
                print(
                    f"{i + 1}/{len(csv_paths)} WARNING: Missing required fields in {csv_path.name}. Skipping this file."
                )
                continue

            # Calculate the total number of unique source_filename values in the current CSV file
            unique_videos = df["source_filename"].nunique()
            total_video_count += unique_videos

            # Filter early to avoid processing rows that we won't ever care about
            filtered_df = df[df["Tube_prob_conf"] >= 0.2]
            total_videos_with_tube_conf += filtered_df["source_filename"].nunique()

            df_list.append(filtered_df)
            print(f"{i + 1}/{len(csv_paths)} Read {len(filtered_df)} rows from {csv_path.name}"
                  f" Dropped {len(df) - len(filtered_df)} rows with Tube_prob_conf < 0.2")

    if df_list:
        aggregated_df = pd.concat(df_list, ignore_index=True)
    else:
        aggregated_df = pd.DataFrame()

    # Remove duplicate rows
    pre_len = len(aggregated_df)
    aggregated_df = aggregated_df.drop_duplicates()
    post_len = len(aggregated_df)
    print(f"Dropped {pre_len - post_len} duplicate rows from aggregated data")

    # Print full column list
    print(f"Loaded {len(aggregated_df)} rows")
    print(aggregated_df.columns.tolist())

    # Ensure device_name is present
    # Add device_name which is missing from reprocessed data
    import sys
    sys.path.append('../rpi')
    from my_fleet_config import INVENTORY
    from expidite_rpi.core.device_config_objects import DeviceCfg
    devices: list[DeviceCfg] = INVENTORY
    device_id_to_name_map = {device.device_id: device.name for device in devices}
    aggregated_df["device_name"] = aggregated_df["device_id"].map(device_id_to_name_map)

    # Extract the RPI number from the end of the device name (the characters after the last -)
    aggregated_df["device_name"] = aggregated_df["device_name"].astype(str)  # Ensure device_name is a string
    aggregated_df["rpi_number"] = aggregated_df["device_name"].str.extract(r"-(\d+)$").astype(int)

    # Extract any rows that don't have a valid RPI number (i.e., the rpi_number is NaN) and print a warning with the device_name and device_id
    invalid_rpi_rows = aggregated_df[aggregated_df["rpi_number"].isna()]
    if not invalid_rpi_rows.empty:
        print("WARNING: Found rows with invalid RPI number (NaN). These rows will be dropped:")
        print(invalid_rpi_rows[["device_name", "device_id"]])
        aggregated_df = aggregated_df.dropna(subset=["rpi_number"])

    # Set the RPI type to 4 if the rpi_number is 10 or less
    aggregated_df["rpi_type"] = aggregated_df["rpi_number"].apply(lambda x: "RPI4" if int(x) <= 10 else "RPI5")

    # To make processing faster and easier, we only want to retain the required columns
    aggregated_df = aggregated_df[required_fields + ["device_name", "rpi_number", "rpi_type"]]

    print(f"Total videos processed: {total_video_count}")
    print(f"Total videos with Tube_prob_conf >= 0.2: {total_videos_with_tube_conf}")

    # Create a csv that lists the file names of the unique source_filename values in the aggregated_df
    unique_source_filenames = aggregated_df["source_filename"].unique()
    unique_source_filenames_df = pd.DataFrame(unique_source_filenames, columns=["source_filename"])
    unique_source_filenames_csv = DOWNLOAD_DIR / f"unique_source_filenames_{TYPE_ID}.csv"
    unique_source_filenames_df.to_csv(unique_source_filenames_csv, index=False)
    print(f"Unique source filenames CSV saved to: {unique_source_filenames_csv.resolve()}")

    return aggregated_df

aggregated_df = aggregate_journals()

CSV files available locally: 345
1/345 Read 4499 rows from V3_CAPOSE_d83add1a11c5_20260506.csv.csv Dropped 1774 rows with Tube_prob_conf < 0.2
2/345 Read 4684 rows from V3_CAPOSE_d83add1a11c5_20260507.csv.csv Dropped 5005 rows with Tube_prob_conf < 0.2
3/345 Read 2763 rows from V3_CAPOSE_d83add1a11c5_20260508.csv.csv Dropped 939 rows with Tube_prob_conf < 0.2
4/345 Read 933 rows from V3_CAPOSE_d83add1a11c5_20260511.csv.csv Dropped 195 rows with Tube_prob_conf < 0.2
5/345 Read 1860 rows from V3_CAPOSE_d83add1a11c5_20260512.csv.csv Dropped 508 rows with Tube_prob_conf < 0.2
6/345 Read 5205 rows from V3_CAPOSE_d83add1a11c5_20260513.csv.csv Dropped 10196 rows with Tube_prob_conf < 0.2
7/345 Read 6437 rows from V3_CAPOSE_d83add1a11c5_20260514.csv.csv Dropped 6082 rows with Tube_prob_conf < 0.2
8/345 Read 3454 rows from V3_CAPOSE_d83add1a11c5_20260515.csv.csv Dropped 1511 rows with Tube_prob_conf < 0.2
9/345 Read 2572 rows from V3_CAPOSE_d83add1a11c5_20260516.csv.csv Dropped 2138 rows with T

In [6]:
# Set tp_feeding to True if tube_prob_conf is greater than 0.5, otherwise set to False
aggregated_df["tp_feeding"] = aggregated_df["Tube_prob_conf"] > 0.5
print(f"Set 'tp_feeding' to True in {aggregated_df['tp_feeding'].sum()} rows, False in {len(aggregated_df) - aggregated_df['tp_feeding'].sum()} rows")

Set 'tp_feeding' to True in 443559 rows, False in 167385 rows


In [7]:
# How many rows don't have a high confidence feeding tube detection (L_tube_conf, R_tube_conf)
aggregated_df["tube_low_confidence"] = (aggregated_df["L_tube_conf"] < 0.5) | (aggregated_df["R_tube_conf"] < 0.5)
print(f"Rows with low confidence feeding tube detection: {aggregated_df['tube_low_confidence'].sum()}")

Rows with low confidence feeding tube detection: 0


In [8]:
# Calculate the number of tubes detected (0, 1, or 2) based on the confidence values of L_tube_conf and R_tube_conf
aggregated_df["num_tubes_detected"] = ((aggregated_df["L_tube_conf"] >= 0.5).astype(int) + (aggregated_df["R_tube_conf"] >= 0.5).astype(int))

# Default the midpoint to MAX_X / 2
aggregated_df["tube_midpoint_x"] = MAX_X / 2

# Calculate the midpoint between the L_tube and R_tube positions if 2 tubes were detected
aggregated_df.loc[aggregated_df["num_tubes_detected"] == 2, "tube_midpoint_x"] = (aggregated_df["L_tube_x"] + aggregated_df["R_tube_x"]) / 2

# Assign all detections to L or R based on their tube_prob_x position relative to the tube_midpoint_x.
# If tube_prob_x is less than tube_midpoint_x, assign to L, otherwise assign to R.
aggregated_df["feeding_from_tube"] = aggregated_df.apply(lambda row: "L" if row["Tube_prob_x"] < row["tube_midpoint_x"] else "R", axis=1)

# Calculate the Y position of the appropriate L or R tube
aggregated_df["feeding_from_tube_y"] = aggregated_df.apply(lambda row: row["L_tube_y"] if row["feeding_from_tube"] == "L" else row["R_tube_y"], axis=1)
aggregated_df["feeding_from_tube_x"] = aggregated_df.apply(lambda row: row["L_tube_x"] if row["feeding_from_tube"] == "L" else row["R_tube_x"], axis=1)

# Calculate the Y distance between the tube_prob detection and the appropriate L or R tube
aggregated_df["tube_prob_to_tube_y_distance"] = (aggregated_df["Tube_prob_y"] - aggregated_df["feeding_from_tube_y"])
aggregated_df["tube_prob_to_tube_x_distance"] = (aggregated_df["Tube_prob_x"] - aggregated_df["feeding_from_tube_x"])

# Calculate the Y distance between the End_prob detection and the appropriate L or R tube
# But set it to NaN if End_prob_conf is less than 0.5, since we don't trust the End_prob detection in that case
aggregated_df["end_prob_to_tube_y_distance"] = (aggregated_df["End_prob_y"] - aggregated_df["feeding_from_tube_y"])
aggregated_df["end_prob_to_tube_x_distance"] = (aggregated_df["End_prob_x"] - aggregated_df["feeding_from_tube_x"])

In [9]:
# Enrich with subject info from subjects.csv
# This uses the RPI number and the frame_start_time to look up the subject info from the subjects.csv file
print("Enriching with subject info from subjects.csv - this can take a few minutes...")
aggregated_df["frame_start_time"] = pd.to_datetime(aggregated_df["frame_start_time"], format="ISO8601")
aggregated_df["date"] = pd.to_datetime(aggregated_df["frame_start_time"], format='ISO8601').dt.date

# get_subject_info returns a dictionary of subject info, so we need to expand that into separate columns
subject_info_df = aggregated_df.apply(
    lambda row: pd.Series(get_subject_info(row["rpi_number"], row["feeding_from_tube"], row["frame_start_time"])), axis=1
)
aggregated_df = pd.concat([aggregated_df, subject_info_df], axis=1)

Enriching with subject info from subjects.csv - this can take a few minutes...


In [10]:
# Save the enriched aggregated CSV for further processing
aggregated_df.to_csv(aggregated_csv_file, index=False)
print(f"Aggregated CSV with {len(aggregated_df)} rows saved to: {aggregated_csv_file.resolve()}")

Aggregated CSV with 610944 rows saved to: C:\Users\bee-ops\code\ChoiceAssay\src\choice_assay\etl\downloads\aggregated_CAPOSE.csv
